In [9]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document
from langchain_groq import ChatGroq


In [2]:
"""
Builds the RAG chain:
    merged retriever -> prompt -> Qwen3-32B on GROQ -> string output
"""

'\nBuilds the RAG chain:\n    merged retriever -> prompt -> Qwen3-32B on GROQ -> string output\n'

In [3]:
SYSTEM_PROMPT="""You are a helpful and professional telecom customer care assistanct.
your job is to help customers resolve technical issues with their mobile service.


use only the context below to answer the customer's question.
the context comes from two sources:
- FAQ entries (general policy and hoe to information)
- Past support tickets (real resolved case with step by step resolutions)

if the context does not contain enough information to answer confidently,say mo clearly and suggest the customer call
611 or use the Mytelcom app.

context:{context}
"""

In [5]:
def _format_docs(docs:list[Document])->str:
    sections=[]
    for doc in docs:
        source = doc.metadata.get("source","unknown").upper()
        sections.append(f"[{source}]\n{doc.page_content}")
    return "\n\n---\n\n".join(sections)

In [19]:

from retriever import build_retriever



def build_chain():
    retriever = build_retriever()

    prompt = ChatPromptTemplate.from_messages([
        ("system",SYSTEM_PROMPT),
        ("human","{question}")
    ])

    llm=ChatGroq(
        model="qwen/qwen3-32b",
        temperature=0,
        max_token=None,
        reasoning_format="parsed",
        timeout=None
    )

    chain=(
        {"context":retriever|_format_docs,"question":RunnablePassthrough()}
        |prompt
        |llm
        |StrOutputParser()
    )

    return chain